# XGBoost Hotel No-Show Model

Train and evaluate an XGBoost classifier using `data/intermediate/noshow_cleaned.csv`.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "intermediate" / "noshow_cleaned.csv"

df = pd.read_csv(DATA_PATH)
df.head()

,booking_id,no_show,branch,booking_month,arrival_month,arrival_day,checkout_month,checkout_day,country,first_time,room,price,platform,num_adults,num_children,price_is_missing,price_matches_expected_pattern,price_currency,price_amount
0,26,0,Orchard,January,November,17,November,18,India,Yes,Null,SGD$ 1405.95,Website,2,0,False,True,SGD,1405.95
1,29,0,Changi,January,July,27,July,29,Australia,Yes,King,USD$ 642.09,Website,2,0,False,True,USD,642.09
2,474,0,Changi,November,November,27,November,28,China,Yes,King,SGD$ 888.83,Email,1,1,False,True,SGD,888.83
3,964,0,Changi,November,December,5,December,7,Indonesia,Yes,King,Null,Phone,2,0,True,False,NaN,NaN
4,1677,0,Changi,February,September,21,September,24,Indonesia,Yes,Null,USD$ 699.67,Website,1,0,False,True,USD,699.67


In [3]:
df.shape, df["no_show"].value_counts(normalize=True).rename("proportion")

((119390, 19),
 no_show
 0    0.629584
 1    0.370416
 Name: proportion, dtype: float64)

Drop identifiers and raw `price` text. Keep engineered price fields such as `price_is_missing`, `price_matches_expected_pattern`, `price_currency`, and `price_amount`.

In [4]:
target_column = "no_show"
drop_columns = ["booking_id", "price", target_column]

X = df.drop(columns=drop_columns)
y = df[target_column]

categorical_features = X.select_dtypes(include=["object", "bool"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object", "bool"]).columns.tolist()

categorical_features, numeric_features

(['branch',
  'booking_month',
  'arrival_month',
  'checkout_month',
  'country',
  'first_time',
  'room',
  'platform',
  'price_is_missing',
  'price_matches_expected_pattern',
  'price_currency'],
 ['arrival_day', 'checkout_day', 'num_adults', 'num_children', 'price_amount'])

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
        (
            "numeric",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
            numeric_features,
        ),
    ]
)

model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ]
)

In [6]:
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tra

In [7]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_proba),
    "average_precision": average_precision_score(y_test, y_proba),
}

metrics

{'accuracy': 0.7339810704414105,
 'balanced_accuracy': 0.6901176409449616,
 'roc_auc': 0.7755323740944973,
 'average_precision': 0.6892961928519501}

In [8]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.75      0.86      0.80     15033
           1       0.69      0.52      0.59      8845

    accuracy                           0.73     23878
   macro avg       0.72      0.69      0.70     23878
weighted avg       0.73      0.73      0.72     23878



In [9]:
confusion_matrix_df = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["actual_show", "actual_no_show"],
    columns=["predicted_show", "predicted_no_show"],
)

confusion_matrix_df

,predicted_show,predicted_no_show
actual_show,12919,2114
actual_no_show,4238,4607


In [10]:
feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
feature_importance = pd.DataFrame(
    {
        "feature": feature_names,
        "importance": pipeline.named_steps["model"].feature_importances_,
    }
).sort_values("importance", ascending=False)

feature_importance.head(20)

,feature,importance
39,categorical__country_China,0.236499
45,categorical__first_time_No,0.089239
0,categorical__branch_Changi,0.067855
1,categorical__branch_Orchard,0.032221
42,categorical__country_Japan,0.027646
51,categorical__room_Single,0.024748
41,categorical__country_Indonesia,0.020274
40,categorical__country_India,0.018886
30,categorical__checkout_month_January,0.017751
8,categorical__booking_month_June,0.016627
